# 10 - Resume training and compare checkpoints

Notebook 08 trained the first Darija GPT checkpoint. This notebook restores the model and AdamW optimizer from that checkpoint, runs additional training steps, and compares validation loss on the same held-out data. It saves the improved checkpoint only when validation loss decreases.

Run notebooks 01, 02, and 08 first. Dataset files, tokenizer, and checkpoint are local artifacts; this notebook does not download data.

![Checkpointing and training-resume flow](../assets/diagrams/10-checkpointing-and-resume.png)

*Validation metrics guide checkpoint selection. This POC keeps the best checkpoint and restores it for the next experiment.*

## 1. Load the existing corpus, tokenizer, and checkpoint

A resumable run needs the exact tokenizer and model configuration used to create the checkpoint. The checksum check prevents accidentally resuming with a different vocabulary.

In [1]:
import copy
import hashlib
import json
import math
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tokenizers import Tokenizer

SEED = 123
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from the darija-llm-lab project directory.")

PROJECT_ROOT = find_project_root()
TOKENIZER_PATH = PROJECT_ROOT / "artifacts/tokenizer/tokenizer_darija.json"
TRAIN_PATH = PROJECT_ROOT / "data/processed/train.jsonl"
VALIDATION_PATH = PROJECT_ROOT / "data/processed/validation.jsonl"
CHECKPOINT_PATH = PROJECT_ROOT / "artifacts/checkpoints/darija_gpt_best.pt"
EVALUATION_PATH = PROJECT_ROOT / "artifacts/evaluation/evaluation.json"

required = [TOKENIZER_PATH, TRAIN_PATH, VALIDATION_PATH, CHECKPOINT_PATH, EVALUATION_PATH]
missing = [str(path.relative_to(PROJECT_ROOT)) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Required files are missing: " + ", ".join(missing) + ". Run notebooks 01, 02, and 08 first."
    )

def read_documents(path):
    with path.open(encoding="utf-8") as stream:
        return [
            row["text"] for line in stream if line.strip()
            for row in [json.loads(line)]
        ]

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.encode_special_tokens = True
eos_token_id = tokenizer.token_to_id("[EOS]")
if eos_token_id is None:
    raise ValueError("The tokenizer must contain the [EOS] token.")

tokenizer_sha256 = hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
if checkpoint["tokenizer_sha256"] != tokenizer_sha256:
    raise ValueError("The checkpoint was trained with a different tokenizer.")
expected_dataset_hashes = {
    "train": hashlib.sha256(TRAIN_PATH.read_bytes()).hexdigest(),
    "validation": hashlib.sha256(VALIDATION_PATH.read_bytes()).hexdigest(),
}
if checkpoint.get("dataset_sha256") != expected_dataset_hashes:
    raise ValueError("The checkpoint uses different train/validation splits. Run notebook 08 again.")
if "optimizer_state_dict" not in checkpoint:
    raise ValueError("This checkpoint has no optimizer state. Run the updated notebook 08 first.")

train_documents = read_documents(TRAIN_PATH)
validation_documents = read_documents(VALIDATION_PATH)
print("Device:", device)
print("Training documents:", len(train_documents))
print("Validation documents:", len(validation_documents))
print("Checkpoint best step:", checkpoint["best_step"])

Device: cpu
Training documents: 2369
Validation documents: 296
Checkpoint best step: 200


## 2. Rebuild the model from its saved configuration

The classes mirror notebook 08 so the state dictionary maps to the same parameter names. The checkpoint supplies the dimensions; this cell does not create new random architecture settings.

In [2]:
@dataclass(frozen=True)
class GPTConfig:
    vocab_size: int
    context_length: int
    embedding_dim: int
    num_layers: int
    num_heads: int
    dropout: float
    qkv_bias: bool = False


class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_heads = config.num_heads
        self.head_dim = config.embedding_dim // config.num_heads
        self.embedding_dim = config.embedding_dim
        self.query = nn.Linear(config.embedding_dim, config.embedding_dim, bias=config.qkv_bias)
        self.key = nn.Linear(config.embedding_dim, config.embedding_dim, bias=config.qkv_bias)
        self.value = nn.Linear(config.embedding_dim, config.embedding_dim, bias=config.qkv_bias)
        self.output = nn.Linear(config.embedding_dim, config.embedding_dim)
        self.attention_dropout = nn.Dropout(config.dropout)
        mask = torch.triu(
            torch.ones(config.context_length, config.context_length, dtype=torch.bool), diagonal=1
        )
        self.register_buffer("future_mask", mask)

    def forward(self, x):
        batch_size, sequence_length, _ = x.shape
        shape = (batch_size, sequence_length, self.num_heads, self.head_dim)
        query = self.query(x).view(shape).transpose(1, 2)
        key = self.key(x).view(shape).transpose(1, 2)
        value = self.value(x).view(shape).transpose(1, 2)
        scores = (query @ key.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(self.future_mask[:sequence_length, :sequence_length], -torch.inf)
        weights = self.attention_dropout(torch.softmax(scores, dim=-1))
        context = weights @ value
        context = context.transpose(1, 2).contiguous().view(
            batch_size, sequence_length, self.embedding_dim
        )
        return self.output(context)


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(config.embedding_dim, 4 * config.embedding_dim),
            nn.GELU(),
            nn.Linear(4 * config.embedding_dim, config.embedding_dim),
        )

    def forward(self, x):
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.embedding_dim)
        self.attention = MultiHeadCausalSelfAttention(config)
        self.norm2 = nn.LayerNorm(config.embedding_dim)
        self.feed_forward = FeedForward(config)
        self.residual_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = x + self.residual_dropout(self.attention(self.norm1(x)))
        x = x + self.residual_dropout(self.feed_forward(self.norm2(x)))
        return x


class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.embedding_dim)
        self.position_embedding = nn.Embedding(config.context_length, config.embedding_dim)
        self.embedding_dropout = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(*(TransformerBlock(config) for _ in range(config.num_layers)))
        self.final_norm = nn.LayerNorm(config.embedding_dim)
        self.output_head = nn.Linear(config.embedding_dim, config.vocab_size, bias=False)
        self.apply(self._initialize_weights)

    @staticmethod
    def _initialize_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, token_ids):
        batch_size, sequence_length = token_ids.shape
        if sequence_length > self.config.context_length:
            raise ValueError("Input sequence exceeds context length.")
        positions = torch.arange(sequence_length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)[None, :, :]
        x = self.embedding_dropout(x)
        return self.output_head(self.final_norm(self.blocks(x)))


config = GPTConfig(**checkpoint["model_config"])
model = GPTModel(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
print("Restored parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Model configuration:", config)

Restored parameters: 359,808
Model configuration: GPTConfig(vocab_size=2000, context_length=64, embedding_dim=64, num_layers=2, num_heads=4, dropout=0.1, qkv_bias=False)


## 3. Recreate document-safe training windows

The same context length and stride as notebook 08 preserve comparability. Windows are made separately for each document, with an end-of-document token appended before they are sliced.

In [3]:
class DarijaGPTDataset(Dataset):
    def __init__(self, documents, tokenizer, context_length, stride, eos_token_id):
        self.examples = []
        for document in documents:
            token_ids = tokenizer.encode(document).ids + [eos_token_id]
            for start in range(0, len(token_ids) - context_length, stride):
                window = token_ids[start:start + context_length + 1]
                if len(window) == context_length + 1:
                    self.examples.append(torch.tensor(window, dtype=torch.long))
        if not self.examples:
            raise ValueError("No complete context windows; check the corpus and context length.")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        window = self.examples[index]
        return window[:-1], window[1:]


class SlidingEvaluationDataset(Dataset):
    """Score each next-token target once while retaining overlapping context."""
    def __init__(self, documents, tokenizer, context_length, stride, eos_token_id):
        if context_length < 2 or stride < 1 or stride > context_length:
            raise ValueError("Use a positive stride no larger than context_length.")
        self.examples = []
        pad_token_id = tokenizer.token_to_id("[PAD]")
        if pad_token_id is None:
            raise ValueError("The tokenizer must contain [PAD].")
        for document in documents:
            token_ids = tokenizer.encode(document).ids + [eos_token_id]
            covered_until = 0
            start = 0
            while start < len(token_ids) - 1:
                end = min(start + context_length + 1, len(token_ids))
                inputs = token_ids[start:end - 1]
                targets = token_ids[start + 1:end]
                labels = [target if start + offset + 1 > covered_until else -100
                          for offset, target in enumerate(targets)]
                pad_count = context_length - len(inputs)
                inputs += [pad_token_id] * pad_count
                labels += [-100] * pad_count
                if any(label != -100 for label in labels):
                    self.examples.append((
                        torch.tensor(inputs, dtype=torch.long),
                        torch.tensor(labels, dtype=torch.long),
                    ))
                covered_until = max(covered_until, end - 1)
                if end == len(token_ids):
                    break
                start += stride
        if not self.examples:
            raise ValueError("No evaluation targets were created.")

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


CONTEXT_LENGTH = config.context_length
BATCH_SIZE = 8
WINDOW_STRIDE = CONTEXT_LENGTH // 2
train_dataset = DarijaGPTDataset(
    train_documents, tokenizer, CONTEXT_LENGTH, WINDOW_STRIDE, eos_token_id
)
validation_dataset = SlidingEvaluationDataset(
    validation_documents, tokenizer, CONTEXT_LENGTH, WINDOW_STRIDE, eos_token_id
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print("Training windows:", len(train_dataset))
print("Validation windows:", len(validation_dataset))

Training windows: 33275
Validation windows: 4411


## 4. Continue from the saved model and optimizer states

Restoring both the model and AdamW state keeps the optimizer moments from the original run. The best validation checkpoint and its history are retained; the resumed run adds new measurements and restores the best weights seen so far.

In [4]:
class CheckpointResumer:
    def __init__(self, model, train_loader, validation_loader, device, checkpoint):
        self.model = model
        self.train_loader = train_loader
        self.validation_loader = validation_loader
        self.device = device
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
        self.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        self.global_step = checkpoint["best_step"]
        self.best_step = checkpoint["best_step"]
        self.best_validation_loss = checkpoint["training"]["best_validation_loss"]
        self.best_model_state = copy.deepcopy(checkpoint["model_state_dict"])
        self.best_optimizer_state = copy.deepcopy(checkpoint["optimizer_state_dict"])
        self.history = list(checkpoint.get("history", []))

    @torch.no_grad()
    def evaluate(self, loader):
        was_training = self.model.training
        self.model.eval()
        total_loss = 0.0
        total_tokens = 0
        for inputs, targets in loader:
            inputs, targets = inputs.to(self.device), targets.to(self.device)
            logits = self.model(inputs)
            total_loss += F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), targets.reshape(-1),
                reduction="sum", ignore_index=-100,
            ).item()
            total_tokens += targets.ne(-100).sum().item()
        self.model.train(was_training)
        if not total_tokens:
            raise ValueError("The evaluation loader produced no target tokens.")
        return total_loss / total_tokens

    def run(self, additional_steps=100, eval_interval=25, gradient_clip=1.0):
        if additional_steps < 1 or eval_interval < 1:
            raise ValueError("additional_steps and eval_interval must be positive.")
        stop_step = self.global_step + additional_steps
        new_history = []

        while self.global_step < stop_step:
            saw_batch = False
            for inputs, targets in self.train_loader:
                saw_batch = True
                self.model.train()
                inputs, targets = inputs.to(self.device), targets.to(self.device)
                self.optimizer.zero_grad(set_to_none=True)
                logits = self.model(inputs)
                loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), gradient_clip)
                self.optimizer.step()
                self.global_step += 1

                if self.global_step % eval_interval == 0 or self.global_step == stop_step:
                    validation_loss = self.evaluate(self.validation_loader)
                    row = {"step": self.global_step, "validation_loss": validation_loss}
                    new_history.append(row)
                    self.history.append(row)
                    print(f"Step {self.global_step:>4} | validation loss {validation_loss:.4f}")
                    if validation_loss < self.best_validation_loss:
                        self.best_validation_loss = validation_loss
                        self.best_step = self.global_step
                        self.best_model_state = {
                            key: value.detach().cpu().clone()
                            for key, value in self.model.state_dict().items()
                        }
                        self.best_optimizer_state = copy.deepcopy(self.optimizer.state_dict())

                if self.global_step >= stop_step:
                    break
            if not saw_batch:
                raise ValueError("The training loader produced no batches.")

        self.model.load_state_dict(self.best_model_state)
        self.optimizer.load_state_dict(self.best_optimizer_state)
        return new_history


resumer = CheckpointResumer(model, train_loader, validation_loader, device, checkpoint)
checkpoint_validation_loss = resumer.evaluate(validation_loader)
print(f"Restored checkpoint validation loss: {checkpoint_validation_loss:.4f}")
print("Resume starts from step:", resumer.global_step)

Restored checkpoint validation loss: 6.7090
Resume starts from step: 200


## 5. Run an additional training segment

The default adds 100 optimizer steps. Keep the validation set and tokenizer unchanged so this run can be compared with the saved checkpoint. Adjust the step count for your hardware; the notebook does not download data or rebuild the tokenizer.

In [5]:
ADDITIONAL_STEPS = 100
EVAL_INTERVAL = 25
resume_history = resumer.run(
    additional_steps=ADDITIONAL_STEPS,
    eval_interval=EVAL_INTERVAL,
)
resumed_validation_loss = resumer.evaluate(validation_loader)
resumed_perplexity = math.exp(resumed_validation_loss) if resumed_validation_loss < 20 else float("inf")
print(f"Best resumed checkpoint loss: {resumed_validation_loss:.4f}")
print(f"Best resumed checkpoint perplexity: {resumed_perplexity:.2f}")
print("Selected checkpoint step:", resumer.best_step)

Step  225 | validation loss 6.7058


Step  250 | validation loss 6.7045


Step  275 | validation loss 6.7025


Step  300 | validation loss 6.6996


Best resumed checkpoint loss: 6.6996
Best resumed checkpoint perplexity: 812.08
Selected checkpoint step: 300


## 6. Compare the checkpoints and save the resumed run

Both losses use the same validation documents, tokenizer, and context-window construction. The new checkpoint replaces the old one only if continued training improves validation loss.

In [6]:
old_best_loss = checkpoint["training"]["best_validation_loss"]
improved = resumed_validation_loss < old_best_loss
comparison = {
    "previous_best_step": checkpoint["best_step"],
    "previous_validation_loss": old_best_loss,
    "resumed_best_step": resumer.best_step,
    "resumed_validation_loss": resumed_validation_loss,
    "resumed_perplexity": resumed_perplexity,
    "validation_loss_improved": improved,
    "additional_steps": ADDITIONAL_STEPS,
}
print(json.dumps(comparison, indent=2))

if improved:
    checkpoint["model_state_dict"] = {
        key: value.detach().cpu() for key, value in model.state_dict().items()
    }
    checkpoint["optimizer_state_dict"] = resumer.optimizer.state_dict()
    checkpoint["best_step"] = resumer.best_step
    checkpoint["completed_steps"] = resumer.global_step
    checkpoint["history"] = resumer.history
    checkpoint["training"]["best_validation_loss"] = resumed_validation_loss
    checkpoint["training"]["validation_perplexity"] = resumed_perplexity
    checkpoint["training"]["resume_steps"] = ADDITIONAL_STEPS
    torch.save(checkpoint, CHECKPOINT_PATH)
    print("Updated best checkpoint:", CHECKPOINT_PATH.relative_to(PROJECT_ROOT))
else:
    print("The existing best checkpoint is kept.")

experiment_path = PROJECT_ROOT / "artifacts/evaluation/resume_comparison.json"
experiment_path.write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print("Saved comparison:", experiment_path.relative_to(PROJECT_ROOT))

{
  "previous_best_step": 200,
  "previous_validation_loss": 6.708978653032442,
  "resumed_best_step": 300,
  "resumed_validation_loss": 6.699598477649619,
  "resumed_perplexity": 812.0796915505088,
  "validation_loss_improved": true,
  "additional_steps": 100
}
Updated best checkpoint: artifacts\checkpoints\darija_gpt_best.pt
Saved comparison: artifacts\evaluation\resume_comparison.json


## Interpretation

A lower validation loss means the resumed checkpoint predicts held-out next tokens better under this tokenizer and split. If it does not improve, keep the earlier checkpoint and investigate data quality, tokenization, learning-rate choice, or model capacity before extending training further.